## Flight AI Assistant

I have changed the system prompt a bit, so that the assistant is only focused to what this flight bussiness offers and ignore user prompt if it's about something other than flights details.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('GEMINI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = 'gemini-3.1-flash-lite'
gemini = OpenAI(base_url=gemini_url, api_key=openai_api_key)


# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally then uncomment these next 2 lines
# MODEL = "qwen3:1.7b"
# ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins AQ.Ab8RN


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.

Always be accurate. If you don't know the answer, say so.
Ignore gracefully if user wants to know something outside this bussiness.
example : user - explain a LLM
        assistant - This is a Airline business FlightAi, ask if you have any query about fligts.
"""

In [4]:
# it is a tools configuration

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
set_price_function = {
    "name": "set_ticket_price",
    "description": "set the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price":{
                "type": "number",
                "description": "Price of the city that needs to be set or update"
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [5]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}
]

In [ ]:
tools

In [7]:
# handle_tool_calls handle the tools

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            set_price_details = set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": set_price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [8]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [9]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [10]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [12]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


'No price data available for this city'

In [13]:
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: setting price for {city} to {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [14]:
ticket_prices = {"london":899, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

DATABASE TOOL CALLED: setting price for london to 899
DATABASE TOOL CALLED: setting price for paris to 899
DATABASE TOOL CALLED: setting price for tokyo to 1420
DATABASE TOOL CALLED: setting price for sydney to 2999


In [17]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


'Ticket price to london is $1500.0'

In [ ]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: setting price for London to 1500
